<a href="https://colab.research.google.com/github/lbruner954/bruner-python-security-project/blob/main/TCPPortScanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
password = input("Enter a password: ")

has_upper = any(c.isupper() for c in password)
has_lower = any(c.islower() for c in password)
has_digit = any(c.isdigit() for c in password)

if len(password) >= 8 and has_upper and has_lower and has_digit:
    print("Strong password")
else:
    print("Weak password")


KeyboardInterrupt: Interrupted by user

In [12]:
#!/usr/bin/env python3
"""
Educational multi-threaded TCP port scanner.

Use only against systems you own or are authorized to assess.
Produces:
  - scan_report.md
  - scan_report.json
"""

from __future__ import annotations

import argparse
import concurrent.futures
import datetime as dt
import ipaddress
import json
import re
import socket
import ssl
import sys
from dataclasses import asdict, dataclass
from typing import Iterable


COMMON_SERVICES = {
    20: "FTP-data",
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    111: "RPCbind",
    135: "MSRPC",
    139: "NetBIOS",
    143: "IMAP",
    161: "SNMP",
    389: "LDAP",
    443: "HTTPS",
    445: "SMB",
    587: "SMTP submission",
    636: "LDAPS",
    993: "IMAPS",
    995: "POP3S",
    1433: "Microsoft SQL Server",
    1521: "Oracle Database",
    2049: "NFS",
    2375: "Docker API",
    3306: "MySQL",
    3389: "RDP",
    5432: "PostgreSQL",
    5900: "VNC",
    6379: "Redis",
    8080: "HTTP alternate",
    8443: "HTTPS alternate",
    9200: "Elasticsearch",
    27017: "MongoDB",
}

REMEDIATIONS = {
    "FTP": (
        "High",
        "Replace FTP with SFTP or FTPS; disable anonymous access; restrict source networks; "
        "rotate credentials and review transferred data."
    ),
    "Telnet": (
        "Critical",
        "Disable Telnet and use SSH with key-based authentication. Restrict SSH exposure "
        "with firewall rules and enforce MFA where available."
    ),
    "SSH": (
        "Medium",
        "Allow SSH only from approved administration networks, disable password/root login, "
        "use key-based authentication, patch OpenSSH, and monitor authentication logs."
    ),
    "HTTP": (
        "Medium",
        "Redirect HTTP to HTTPS where appropriate, patch the web server and application, "
        "remove unnecessary headers, and restrict management interfaces."
    ),
    "HTTPS": (
        "Medium",
        "Use a valid certificate, disable obsolete TLS versions and weak ciphers, patch the "
        "web stack, enable security headers, and review exposed application endpoints."
    ),
    "SMB": (
        "High",
        "Block SMB from untrusted networks, disable SMBv1, require signing where appropriate, "
        "patch the host, and restrict shares and privileges."
    ),
    "RDP": (
        "High",
        "Place RDP behind a VPN or zero-trust gateway, require MFA, restrict source IPs, "
        "enable Network Level Authentication, and apply current patches."
    ),
    "MySQL": (
        "High",
        "Do not expose the database publicly. Restrict it to application networks, require "
        "strong authentication, encrypt connections, and patch the database."
    ),
    "PostgreSQL": (
        "High",
        "Restrict PostgreSQL to approved application/admin networks, use TLS and strong "
        "authentication, minimize privileges, and keep the server patched."
    ),
    "Redis": (
        "Critical",
        "Block public access immediately, bind Redis to trusted interfaces, require "
        "authentication, use ACLs/TLS, and check for unauthorized data or commands."
    ),
    "MongoDB": (
        "Critical",
        "Restrict MongoDB to trusted networks, enable authentication and authorization, "
        "use TLS, patch it, and inspect logs for unauthorized access."
    ),
    "Docker API": (
        "Critical",
        "Never expose an unauthenticated Docker API. Disable TCP exposure or protect it with "
        "mutual TLS and strict network controls."
    ),
    "default": (
        "Review",
        "Confirm that the service is required. Patch it, restrict access with host/network "
        "firewalls, enforce authentication and encryption, and remove the service if unused."
    ),
}


@dataclass
class Finding:
    host: str
    port: int
    service: str
    state: str
    banner: str
    version: str
    severity: str
    remediation: str
    error: str = ""


def clean_text(value: bytes, limit: int = 512) -> str:
    text = value[:limit].decode("utf-8", errors="replace")
    return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text).strip()


def identify_version(banner: str, service: str) -> str:
    if not banner:
        return "Not disclosed"

    patterns = [
        r"\b(?:OpenSSH|Apache|nginx|Microsoft-IIS|vsftpd|ProFTPD|Postfix|Exim|"
        r"Redis|MongoDB|MySQL|PostgreSQL|SQLite|Samba)[/\s-]*([0-9]+(?:\.[0-9]+){1,3})",
        r"\b([0-9]+\.[0-9]+(?:\.[0-9]+)?)\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, banner, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    return "Unknown"


def service_metadata(service: str) -> tuple[str, str]:
    for key, value in REMEDIATIONS.items():
        if key.lower() in service.lower():
            return value
    return REMEDIATIONS["default"]


def grab_banner(sock: socket.socket, host: str, port: int, timeout: float) -> str:
    sock.settimeout(timeout)

    # TLS-aware probe for HTTPS-like services.
    if port in {443, 8443}:
        context = ssl.create_default_context()
        context.check_hostname = False
        context.verify_mode = ssl.CERT_NONE

        try:
            with context.wrap_socket(sock, server_hostname=host) as tls_sock:
                request = (
                    f"HEAD / HTTP/1.0\r\n"
                    f"Host: {host}\r\n"
                    f"User-Agent: CSIT2033-Authorized-Audit\r\n\r\n"
                ).encode()
                tls_sock.sendall(request)
                return clean_text(tls_sock.recv(512))
        except (ssl.SSLError, socket.timeout, OSError):
            return ""

    # Some services send a banner without prompting.
    try:
        data = sock.recv(512)
        if data:
            return clean_text(data)
    except socket.timeout:
        pass

    # A small, non-destructive HTTP probe for web services.
    if port in {80, 8080}:
        try:
            request = (
                f"HEAD / HTTP/1.0\r\n"
                f"Host: {host}\r\n"
                f"User-Agent: CSIT2033-Authorized-Audit\r\n\r\n"
            ).encode()
            sock.sendall(request)
            return clean_text(sock.recv(512))
        except (socket.timeout, OSError):
            return ""

    return ""


def scan_one(host: str, port: int, timeout: float) -> Finding | None:
    service = COMMON_SERVICES.get(port, "Unknown TCP service")

    try:
        sock = socket.create_connection((host, port), timeout=timeout)
    except (socket.timeout, ConnectionRefusedError):
        return None
    except OSError as exc:
        return Finding(
            host=host,
            port=port,
            service=service,
            state="error",
            banner="",
            version="Unknown",
            severity="Unknown",
            remediation="Investigate the connection error and verify authorization/routing.",
            error=str(exc),
        )

    with sock:
        banner = grab_banner(sock, host, port, timeout)

    version = identify_version(banner, service)
    severity, remediation = service_metadata(service)

    return Finding(
        host=host,
        port=port,
        service=service,
        state="open",
        banner=banner,
        version=version,
        severity=severity,
        remediation=remediation,
    )


def parse_ports(spec: str) -> list[int]:
    ports: set[int] = set()

    for item in spec.split(","):
        item = item.strip()

        if "-" in item:
            start_text, end_text = item.split("-", 1)
            start, end = int(start_text), int(end_text)
            if start > end:
                start, end = end, start
            ports.update(range(start, end + 1))
        else:
            ports.add(int(item))

    if not ports or any(port < 1 or port > 65535 for port in ports):
        raise ValueError("Ports must be between 1 and 65535.")

    return sorted(ports)


def parse_targets(target: str, max_hosts: int) -> list[str]:
    # CIDR network, such as 192.168.1.0/28.
    if "/" in target:
        network = ipaddress.ip_network(target, strict=False)
        hosts = [str(address) for address in network.hosts()]
    else:
        # Single IP address.
        try:
            address = ipaddress.ip_address(target)
            hosts = [str(address)]
        except ValueError:
            # Start-end IPv4 range, such as 192.168.1.10-192.168.1.20.
            if "-" in target:
                start_text, end_text = target.split("-", 1)
                start = ipaddress.ip_address(start_text.strip())
                end = ipaddress.ip_address(end_text.strip())

                if start.version != 4 or end.version != 4 or int(start) > int(end):
                    raise ValueError("Only ascending IPv4 start-end ranges are supported.")

                hosts = [
                    str(ipaddress.ip_address(value))
                    for value in range(int(start), int(end) + 1)
                ]
            else:
                # Domain name. Resolve it once before scanning.
                try:
                    resolved = socket.getaddrinfo(
                        target, None, type=socket.SOCK_STREAM
                    )
                    hosts = sorted({entry[4][0] for entry in resolved})
                except socket.gaierror as exc:
                    raise ValueError(f"Could not resolve target: {exc}") from exc

    if not hosts:
        raise ValueError("The target produced no usable host addresses.")

    if len(hosts) > max_hosts:
        raise ValueError(
            f"Target contains {len(hosts)} hosts, exceeding --max-hosts {max_hosts}."
        )

    return hosts


def write_reports(findings: list[Finding], target: str, ports: str, timeout: float) -> None:
    timestamp = dt.datetime.now(dt.timezone.utc).isoformat()
    records = [asdict(finding) for finding in findings]

    report = {
        "scan_metadata": {
            "target": target,
            "ports": ports,
            "timeout_seconds": timeout,
            "completed_utc": timestamp,
            "authorization_note": (
                "Run only against assets owned by or explicitly authorized by the tester."
            ),
        },
        "findings": records,
    }

    with open("scan_report.json", "w", encoding="utf-8") as output:
        json.dump(report, output, indent=2)

    with open("scan_report.md", "w", encoding="utf-8") as output:
        output.write("# TCP Port Scan Report\n\n")
        output.write(f"- Target: `{target}`\n")
        output.write(f"- Ports: `{ports}`\n")
        output.write(f"- Timeout: `{timeout}` seconds\n")
        output.write(f"- Completed: `{timestamp}`\n\n")

        if not findings:
            output.write("No open TCP services were identified.\n")
            return

        output.write("## Exposed Services\n\n")
        output.write(
            "| Host | Port | Service | Version | Severity | Banner |\n"
            "|---|---:|---|---|---|---|\n"
        )

        for finding in findings:
            banner = finding.banner.replace("|", "\\|").replace("\n", " ")
            output.write(
                f"| {finding.host} | {finding.port} | {finding.service} | "
                f"{finding.version} | {finding.severity} | {banner} |\n"
            )

        output.write("\n## Remediation Plan\n\n")

        for number, finding in enumerate(findings, start=1):
            output.write(
                f"### {number}. {finding.host}:{finding.port} — "
                f"{finding.service}\n\n"
            )
            output.write(f"- Severity: **{finding.severity}**\n")
            output.write(f"- Observed version: `{finding.version}`\n")
            output.write(f"- Recommended action: {finding.remediation}\n")
            output.write(
                "- Validation: Re-scan after remediation and confirm that the port is "
                "closed or restricted to the approved source networks.\n\n"
            )


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="Authorized multi-threaded TCP port scanner with banner reporting."
    )
    parser.add_argument(
        "target",
        help=(
            "Domain, IP, CIDR range, or IPv4 start-end range "
            "(example: 192.168.1.10-192.168.1.20)"
        ),
    )
    parser.add_argument(
        "-p",
        "--ports",
        default="1-1024",
        help="Ports, comma-separated values and ranges; default: 1-1024",
    )
    parser.add_argument(
        "-t",
        "--timeout",
        type=float,
        default=1.0,
        help="TCP connection/banner timeout in seconds; default: 1.0",
    )
    parser.add_argument(
        "-w",
        "--workers",
        type=int,
        default=32,
        help="Maximum concurrent worker threads; default: 32",
    )
    parser.add_argument(
        "--max-hosts",
        type=int,
        default=256,
        help="Safety limit for target hosts; default: 256",
    )

    if argv is None:
        # Filter out Jupyter/IPython notebook specific arguments if present
        # This is necessary because in Colab, sys.argv often contains '-f' and a connection file.
        # We only consider arguments after the script name (sys.argv[0])
        processed_argv = sys.argv[1:]
        # Remove '-f' and its value if present, which is common in Colab/IPython environments
        try:
            while True:
                f_index = processed_argv.index('-f')
                # Remove '-f' and the argument immediately following it
                del processed_argv[f_index : f_index + 2]
        except ValueError:
            pass # '-f' not found, proceed with the current processed_argv
        argv = processed_argv

    args = parser.parse_args(argv)

    if args.timeout <= 0:
        parser.error("--timeout must be greater than zero.")

    if args.workers < 1 or args.workers > 128:
        parser.error("--workers must be between 1 and 128.")

    try:
        ports = parse_ports(args.ports)
        hosts = parse_targets(args.target, args.max_hosts)
    except ValueError as exc:
        parser.error(str(exc))

    jobs = [(host, port) for host in hosts for port in ports]
    findings: list[Finding] = []

    print(f"Scanning {len(hosts)} host(s) and {len(ports)} port(s)...")

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=args.workers
    ) as executor:
        future_map = {
            executor.submit(scan_one, host, port, args.timeout): (host, port)
            for host, port in jobs
        }

        for future in concurrent.futures.as_completed(future_map):
            host, port = future_map[future]

            try:
                finding = future.result()
            except Exception as exc:
                print(f"[!] Worker error for {host}:{port}: {exc}", file=sys.stderr)
                continue

            if finding is not None:
                findings.append(finding)
                print(
                    f"[+] {finding.host}:{finding.port} open "
                    f"({finding.service}, version={finding.version})"
                )

    findings.sort(key=lambda item: (item.host, item.port))
    write_reports(findings, args.target, args.ports, args.timeout)

    print(f"\nCompleted. Open services found: {len(findings)}")
    print("Created: scan_report.md and scan_report.json")
    return 0


if __name__ == "__main__":
    raise SystemExit(main(['192.168.56.101']))


Scanning 1 host(s) and 1024 port(s)...

Completed. Open services found: 0
Created: scan_report.md and scan_report.json


SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
